In [ ]:
import sys
import os

# Add project root to path so `src_py` is importable as a package
_root = os.path.normpath(os.path.join(os.path.abspath(''), '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np
import matplotlib.pyplot as plt

from src_py import surrogate_model as srg, muvec as srg_grid
from src_py import surrogate_model_proj as srg_proj


In [ ]:
alpha = 4
gamma = 2.0
phi = 2.5
srg_vcut = srg(alpha, gamma, phi)
srg_proj_vcut = srg_proj(0.0, alpha, gamma, phi)

mu_max = 10
Nmu = 20
srg_proj_grid = np.linspace(0, mu_max, Nmu)

print(f"Surrogate model vcut: {srg_vcut}")
print(f"Surrogate projection vcut: {srg_proj_vcut}")

In [ ]:
figure, ax = plt.subplots(figsize=(5, 3.5))

vcut = srg_proj(srg_proj_grid, alpha, gamma, phi)
ax.plot(vcut, srg_proj_grid, marker='o', label='Surrogate Projection')
if srg_vcut is not None:
    ax.plot(srg_vcut, srg_grid, color='r', linestyle='--', label='Surrogate Model')
ax.set_xlabel(r'$v_{\mathrm{cut}}$')
ax.set_ylabel(r'$\mu$')
ax.set_title(f'Surrogate Projection vs Model (α={alpha}, γ={gamma}, φ={phi})')
ax.legend()
plt.tight_layout()

In [ ]:
from src_py import generate_c_code

generate_c_code(
    nn_model      = "../model/nn_model.pth",
    svm_model     = "../model/svm_model.pkl",
    normalization = "../model/normalization.npz",
    output_dir    = "../generated_c_code"
)


In [ ]:
import subprocess
import numpy as np

# --- Run the compiled C test program and capture output ---
proc = subprocess.run(
    ["./test_surrogate"],
    cwd="../generated_c_code",
    capture_output=True, text=True
)
c_out = np.array([
    float(line.split("=")[1])
    for line in proc.stdout.strip().splitlines()
])

# --- Python reference (same inputs as test_surrogate.c: alpha=4, gamma=1.0, phi=2.5) ---
py_out = srg(4, 1.0, 2.5)

# --- Comparison ---
abs_err = np.abs(c_out - py_out)
print(f"{'i':>3}  {'C output':>14}  {'Python output':>14}  {'abs error':>12}")
print("-" * 50)
for i, (c, p, e) in enumerate(zip(c_out, py_out, abs_err)):
    print(f"{i:>3}  {c:>14.8f}  {p:>14.8f}  {e:>12.2e}")

print(f"\nMax absolute error : {abs_err.max():.2e}")
print(f"Mean absolute error: {abs_err.mean():.2e}")
